# XGBoost

In [1]:
import pandas as pd
import xgboost as xgb
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.metrics import accuracy_score

print("## 🚀 Fase 3: Activando el Modo XGBoost")

# --- PASO 1: Cargar Datos Limpios (Train) ---
df = pd.read_csv('titanic_train_clean.csv')
X = df.drop('Survived', axis=1)
y = df['Survived']

# División Train/Test interna para validar
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# --- PASO 2: Configurar la Búsqueda de Hiperparámetros (GridSearch) ---
# XGBoost tiene parámetros muy sensibles. Vamos a buscar la combinación ganadora.
model = xgb.XGBClassifier(objective='binary:logistic', 
                          eval_metric='logloss', 
                          use_label_encoder=False, 
                          random_state=42)

param_grid = {
    'n_estimators': [100, 200, 300],    # Cuántos árboles correctivos crear
    'learning_rate': [0.01, 0.05, 0.1], # Qué tan rápido aprende (menor es más preciso pero lento)
    'max_depth': [3, 4, 5],             # Profundidad (XGBoost prefiere árboles pequeños)
    'subsample': [0.8, 1.0]             # ¿Usar todos los datos o una fracción para evitar overfitting?
}

print("⏳ Iniciando optimización de XGBoost (esto puede tomar unos minutos)...")
grid_search = GridSearchCV(model, param_grid, cv=5, n_jobs=-1, verbose=1)
grid_search.fit(X_train, y_train)

# --- PASO 3: Resultados ---
best_xgb = grid_search.best_estimator_
print(f"\n🏆 MEJOR CONFIGURACIÓN: {grid_search.best_params_}")
print(f"🚀 Precisión en Validación Cruzada: {grid_search.best_score_:.2%}")

# Validación final con el set de prueba interno
final_acc = accuracy_score(y_test, best_xgb.predict(X_test))
print(f"🎯 Precisión Real (Test Interno): {final_acc:.2%}")

## 🚀 Fase 3: Activando el Modo XGBoost
⏳ Iniciando optimización de XGBoost (esto puede tomar unos minutos)...
Fitting 5 folds for each of 54 candidates, totalling 270 fits


/home/puya-chilensis/data-science/env/lib/python3.12/site-packages/xgboost/training.py:199: UserWarning: [20:38:24] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/home/puya-chilensis/data-science/env/lib/python3.12/site-packages/xgboost/training.py:199: UserWarning: [20:38:24] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/home/puya-chilensis/data-science/env/lib/python3.12/site-packages/xgboost/training.py:199: UserWarning: [20:38:24] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/home/puya-chilensis/data-science/env/lib/python3.12/site-packages/xgboost/training.py:199: UserWarning: [20:38:24] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fo


🏆 MEJOR CONFIGURACIÓN: {'learning_rate': 0.05, 'max_depth': 3, 'n_estimators': 200, 'subsample': 0.8}
🚀 Precisión en Validación Cruzada: 84.27%
🎯 Precisión Real (Test Interno): 84.92%


/home/puya-chilensis/data-science/env/lib/python3.12/site-packages/xgboost/training.py:199: UserWarning: [20:38:28] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


In [2]:
print("## 🚀 ETAPA FINAL: Generando Predicciones con XGBoost")

# 1. Cargar el archivo de prueba
test_df = pd.read_csv('test.csv')
passenger_ids = test_df['PassengerId']

# --- PIPELINE DE TRANSFORMACIÓN (Idéntico al de Train) ---
# Es vital que hagamos EXACTAMENTE lo mismo que hicimos para entrenar

# A. Títulos
test_df['Title'] = test_df['Name'].str.extract(r' ([A-Za-z]+)\.', expand=False)
test_df['Title'] = test_df['Title'].replace({'Mlle': 'Miss', 'Ms': 'Miss', 'Mme': 'Mrs'})
rare_titles = ['Dr', 'Rev', 'Col', 'Major', 'Capt', 'Lady', 'Countess', 'Sir', 'Don', 'Jonkheer', 'Dona']
for title in rare_titles:
    mask = (test_df['Title'] == title)
    test_df.loc[mask & (test_df['Sex'] == 'male'), 'Title'] = 'Mr'
    test_df.loc[mask & (test_df['Sex'] == 'female'), 'Title'] = 'Mrs'

# B. Edad (Imputación)
test_df['Age'] = test_df.groupby('Title')['Age'].transform(lambda x: x.fillna(x.median()))

# C. Familia
test_df['FamilySize'] = test_df['SibSp'] + test_df['Parch'] + 1
test_df['FamilyGroup'] = test_df['FamilySize'].apply(lambda x: 'Alone' if x == 1 else ('Small' if 2 <= x <= 4 else 'Large'))

# D. Fare (Rellenar el nulo y crear rangos)
test_df['Fare'] = test_df['Fare'].fillna(test_df['Fare'].median())
test_df['FareBin'] = pd.qcut(test_df['Fare'], 4, labels=[0, 1, 2, 3])

# E. Encoding (Texto a Números)
test_df['Sex_Code'] = test_df['Sex'].map({'male': 0, 'female': 1}).astype(int)
test_df['Embarked_Code'] = test_df['Embarked'].map({'S': 0, 'C': 1, 'Q': 2}).astype(int)
test_df['Title_Code'] = test_df['Title'].map({'Mr': 0, 'Miss': 1, 'Mrs': 2, 'Master': 3}).astype(int)
test_df['Family_Code'] = test_df['FamilyGroup'].map({'Alone': 0, 'Small': 1, 'Large': 2}).astype(int)

# F. Selección de Columnas (Crucial: El mismo orden que en el entrenamiento)
features = ['Pclass', 'Age', 'FareBin', 'Sex_Code', 'Embarked_Code', 'Title_Code', 'Family_Code']
X_submission = test_df[features]

# --- PREDICCIÓN CON EL SUPER MODELO ---
# Usamos best_xgb (tu modelo de 84.9%)
final_preds = best_xgb.predict(X_submission)

# --- GUARDAR ---
submission = pd.DataFrame({
    'PassengerId': passenger_ids,
    'Survived': final_preds
})

filename = 'Titanic_Submission_XGBoost.csv'
submission.to_csv(filename, index=False)

print(f"\n🎉 ¡MISIÓN CUMPLIDA! Archivo '{filename}' generado.")
print("   Este archivo lleva la potencia del Gradient Boosting.")

## 🚀 ETAPA FINAL: Generando Predicciones con XGBoost

🎉 ¡MISIÓN CUMPLIDA! Archivo 'Titanic_Submission_XGBoost.csv' generado.
   Este archivo lleva la potencia del Gradient Boosting.


In [ ]:
from sklearn.ensemble import VotingClassifier

print("## ⚖️ Estrategia Final: Voting Ensemble (El Comité de Expertos)")

# 1. Recuperamos tus dos mejores guerreros (con los parámetros que ganaron antes)
# Random Forest (El Robusto)
rf_best = RandomForestClassifier(n_estimators=100, max_depth=5, min_samples_leaf=5, min_samples_split=15, random_state=42)

# XGBoost (El Agresivo)
xgb_best = xgb.XGBClassifier(n_estimators=200, learning_rate=0.05, max_depth=3, subsample=0.8, 
                             objective='binary:logistic', eval_metric='logloss', use_label_encoder=False, random_state=42)

# 2. Creamos el "Comité" (VotingClassifier)
# voting='soft' significa que promedia las probabilidades (ej: 0.8 + 0.4 / 2 = 0.6 -> Vive)
voting_model = VotingClassifier(estimators=[
    ('rf', rf_best),
    ('xgb', xgb_best)
], voting='soft')

# 3. Entrenamos al Comité con TODOS los datos (X_train + X_test del split anterior)
# Para la entrega final, es mejor usar todo el dataset disponible
print("⏳ Entrenando el modelo combinado...")
voting_model.fit(X, y) # Usamos X e y completos del inicio

# 4. Predecir sobre test.csv (X_submission ya la tienes cargada del paso anterior)
voting_preds = voting_model.predict(X_submission)

# 5. Generar Archivo
submission = pd.DataFrame({
    'PassengerId': passenger_ids,
    'Survived': voting_preds
})

filename = 'Titanic_Submission_Ensemble.csv'
submission.to_csv(filename, index=False)

print(f"\n🎉 ¡ÚLTIMO INTENTO! Archivo '{filename}' generado.")
print("   Este modelo promedia la sabiduría del Random Forest y la precisión del XGBoost.")